<a href="https://colab.research.google.com/github/trishank18/LMS/blob/main/GEN_AI_WITH_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Step 1: Imports and File Upload
from pathlib import Path
import pandas as pd
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA_FILE = Path("/content/sales_data.csv")

# Upload sales_data.csv if it does not exist already
if not DATA_FILE.is_file():
    print("Choose sales_data.csv in the upload dialog.")
    uploaded_files = files.upload()
    if "sales_data.csv" not in uploaded_files:
        raise FileNotFoundError(
            "sales_data.csv was not uploaded."
        )
else:
    print(f"File already exists: {DATA_FILE.name}")

File already exists: sales_data.csv


In [12]:
# Step 2: Initialize Spark Session
spark = (
    SparkSession.builder
    .appName("Sales Dataset EDA")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
print(f"Spark started. Version: {spark.version}")
print("Adaptive Query Execution:", spark.conf.get("spark.sql.adaptive.enabled"))

Spark started. Version: 4.0.4
Adaptive Query Execution: true


In [13]:
# Step 3: Load and Cache Dataset
sales_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "FAILFAST")
    .csv(str(DATA_FILE))
    .cache()
)

row_count = sales_df.count()
print(f"CSV loaded and cached: {row_count:,} rows.")
sales_df.printSchema()

CSV loaded and cached: 100,000 rows.
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)



In [14]:
# Step 4: Summary Statistics
print("Basic statistics:")
sales_df.describe().show(truncate=False)

Basic statistics:
+-------+-----------------+----------------+---------+---------------+------------------+------------------+------------------+-------+
|summary|transaction_id   |customer_id     |product  |category       |quantity          |price             |revenue           |region |
+-------+-----------------+----------------+---------+---------------+------------------+------------------+------------------+-------+
|count  |100000           |100000          |100000   |100000         |100000            |95000             |95000             |98000  |
|mean   |50000.5          |9999.98057      |NULL     |NULL           |5.49046           |252.99148873684425|1386.3585905263071|NULL   |
|stddev |28867.65779668775|5754.82639433378|NULL     |NULL           |2.8692875973860046|143.08048382063683|1145.3201789786315|NULL   |
|min    |1                |1               |Bookshelf|Clothing       |1                 |5.01              |5.07              |Central|
|max    |100000           |200

In [15]:
# Step 5: Check Missing Values
missing_counts = sales_df.agg(
    *[
        F.sum(
            F.when(
                F.col(f"`{column}`").isNull()
                | (F.trim(F.col(f"`{column}`").cast("string")) == ""),
                1,
            ).otherwise(0)
        ).alias(column)
        for column in sales_df.columns
    ]
)

print("Missing-value counts by column:")
missing_counts.show(truncate=False)

Missing-value counts by column:
+--------------+-----------+-------+--------+--------+-----+-------+----+------+
|transaction_id|customer_id|product|category|quantity|price|revenue|date|region|
+--------------+-----------+-------+--------+--------+-----+-------+----+------+
|0             |0          |0      |0       |0       |5000 |5000   |0   |2000  |
+--------------+-----------+-------+--------+--------+-----+-------+----+------+



In [17]:
# Step 6: Sample and Convert to Pandas for Visualization/Inspection
sample_df = sales_df.sample(
    withReplacement=False,
    fraction=0.001,
    seed=42,
).cache()

sample_count = sample_df.count()
print(f"Sample created: {sample_count:,} rows (approximately 0.1%).")

sample_pd = sample_df.toPandas()
print(f"Converted sample to Pandas: {sample_pd.shape[0]:,} rows.")
display(sample_pd.head())

Sample created: 96 rows (approximately 0.1%).
Converted sample to Pandas: 96 rows.


,transaction_id,customer_id,product,category,quantity,price,revenue,date,region
0,7283,14596,T-Shirt,Clothing,2,20.13,40.26,2023-02-18,South
1,7396,3056,Headphones,Electronics,6,181.67,1090.02,2024-06-10,West
2,7621,545,Stapler,Office Supplies,7,363.40,2543.80,2023-02-08,South
3,8282,8987,Table,Furniture,4,238.17,952.68,2024-10-13,Central
4,8977,623,Stapler,Office Supplies,6,236.30,1417.80,2023-01-19,South


In [1]:
!pip install --upgrade pandas polars dask

In [3]:
import time
import pandas as pd
import polars as pl
from pathlib import Path
from google.colab import files

DATA_FILE = Path("/content/sales_data.csv")
if not DATA_FILE.is_file():
    print("Choose sales_data.csv in the upload dialog.")
    uploaded_files = files.upload()
    if "sales_data.csv" not in uploaded_files:
        raise FileNotFoundError("sales_data.csv was not uploaded.")

In [4]:
# Pandas Benchmark
start_time = time.perf_counter()
pdf = pd.read_csv(DATA_FILE)
pand_result = pdf.groupby("category")["revenue"].sum()
pandas_duration = time.perf_counter() - start_time

print(f"Pandas execution time: {pandas_duration:.4f} seconds")
print(pand_result)

Pandas execution time: 0.1984 seconds
category
Clothing           33294150.09
Electronics        33040666.20
Furniture          32702256.78
Office Supplies    32666993.03
Name: revenue, dtype: float64


In [5]:
# Polars Benchmark
start_time = time.perf_counter()
pldf = pl.read_csv(DATA_FILE)
pol_result = pldf.group_by("category").agg(pl.col("revenue").sum())
polars_duration = time.perf_counter() - start_time

print(f"Polars execution time: {polars_duration:.4f} seconds")
print(pol_result)

speedup = pandas_duration / polars_duration
print(f"\nPolars is {speedup:.2f}x faster than Pandas on this task.")

Polars execution time: 0.0981 seconds
shape: (4, 2)
┌─────────────────┬──────────┐
│ category        ┆ revenue  │
│ ---             ┆ ---      │
│ str             ┆ f64      │
╞═════════════════╪══════════╡
│ Clothing        ┆ 3.3294e7 │
│ Electronics     ┆ 3.3041e7 │
│ Office Supplies ┆ 3.2667e7 │
│ Furniture       ┆ 3.2702e7 │
└─────────────────┴──────────┘

Polars is 2.02x faster than Pandas on this task.


In [2]:
import time
import pandas as pd
import polars as pl
from pathlib import Path
from google.colab import files

# Ensure the data file exists
DATA_FILE = Path("/content/sales_data.csv")
if not DATA_FILE.is_file():
    print("Choose sales_data.csv in the upload dialog.")
    uploaded_files = files.upload()
    if "sales_data.csv" not in uploaded_files:
        raise FileNotFoundError("sales_data.csv was not uploaded.")

def benchmark_pandas(filepath):
    start_time = time.perf_counter()
    # Load CSV
    df = pd.read_csv(filepath)
    # Groupby category, sum revenue
    result = df.groupby("category")["revenue"].sum()
    end_time = time.perf_counter()
    return end_time - start_time

def benchmark_polars(filepath):
    start_time = time.perf_counter()
    # Load CSV
    df = pl.read_csv(filepath)
    # Groupby category, sum revenue
    result = df.group_by("category").agg(pl.col("revenue").sum())
    end_time = time.perf_counter()
    return end_time - start_time

# Warm up and run benchmark
pandas_duration = benchmark_pandas(DATA_FILE)
polars_duration = benchmark_polars(DATA_FILE)

speedup = pandas_duration / polars_duration

print(f"Pandas execution time: {pandas_duration:.4f} seconds")
print(f"Polars execution time: {polars_duration:.4f} seconds")
print(f"Polars is {speedup:.2f}x faster than Pandas on this task.")

Pandas execution time: 0.5227 seconds
Polars execution time: 0.1624 seconds
Polars is 3.22x faster than Pandas on this task.
